In [ ]:
import os, sys, json, zipfile, shutil, random, time, re, math, subprocess, platform
from pathlib import Path
import numpy as np

PROJECT_DIR = Path("/content/booking_agent")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version)
print("Project:", PROJECT_DIR)
!nvidia-smi

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Project: /content/booking_agent
Tue Nov 24 06:34:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |      

In [15]:
from google.colab import files
import zipfile
from pathlib import Path

PROJECT_DIR = Path("/content/booking_agent")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = globals().get('uploaded', {})
if not uploaded:
    uploaded = files.upload()

zip_candidates = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_candidates:
    # Fallback to check directory manually if uploaded is empty
    zip_candidates = [p.name for p in Path("/content").glob("*.zip")]

assert zip_candidates, "Please upload the booking_agent_dataset ZIP file."
ZIP_PATH = Path("/content") / zip_candidates[0]

print(f"Extracting {ZIP_PATH} to {PROJECT_DIR}...")
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(PROJECT_DIR)

# The archive may contain a top-level folder or may extract files directly.
if not (PROJECT_DIR / "data" / "train.json").exists():
    possible = list(PROJECT_DIR.rglob("train.json"))
    assert possible, "Could not find data/train.json inside the uploaded ZIP."
    DATA_ROOT = possible[0].parent
    PROJECT_ROOT = DATA_ROOT.parent
else:
    DATA_ROOT = PROJECT_DIR / "data"
    PROJECT_ROOT = PROJECT_DIR

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("Files:", sorted(p.name for p in PROJECT_ROOT.iterdir()))

Extracting /content/booking_agent_dataset.zip to /content/booking_agent...
PROJECT_ROOT: /content/booking_agent
DATA_ROOT: /content/booking_agent/data
Files: ['README.md', 'catalog.py', 'data', 'generate.py', 'runs', 'stats.py', 'validate.py']


In [ ]:
readme = PROJECT_ROOT / "README.md"
if readme.exists():
    print(readme.read_text(encoding="utf-8"))

for filename in ["train.json", "val.json", "test.json", "tools.json", "taxonomy.json", "stats.json"]:
    path = DATA_ROOT / filename
    print(filename, "exists =", path.exists(), "bytes =", path.stat().st_size if path.exists() else None)

with open(DATA_ROOT / "train.json", encoding="utf-8") as f:
    train_rows = json.load(f)
with open(DATA_ROOT / "val.json", encoding="utf-8") as f:
    val_rows = json.load(f)
with open(DATA_ROOT / "test.json", encoding="utf-8") as f:
    test_rows = json.load(f)
with open(DATA_ROOT / "tools.json", encoding="utf-8") as f:
    tools = json.load(f)

print("Train:", len(train_rows), "Validation:", len(val_rows), "Test:", len(test_rows))
print("Example ID:", train_rows[0]["id"])
print("Meta:", train_rows[0]["meta"])
print("First messages:")
for m in train_rows[0]["messages"][:5]:
    print(m)

# Booking agent SFT dataset (synthetic)

- `data/train.json` 4,800 · `data/val.json` 600 · `data/test.json` 600 conversations
- Each row: `{"id", "messages", "meta"}`. Messages use the HF/OpenAI chat format with `tool_calls`;
  pass `data/tools.json` as `tools=` to `tokenizer.apply_chat_template`.
- `meta`: scenario, specialty, urgency (routine / urgent / emergency), red_flag, expected_tools,
  decision_turn (index of the routing/escalation turn in `messages`), user_style, n_turns.
- Test split is held out: unseen symptom phrasings, unseen prompts, and a later date range.
- `data/taxonomy.json`: symptom → specialty → urgency table used to generate the data. **Needs clinician review.**
- Regenerate: `python generate.py && python validate.py && python stats.py` (seeded, deterministic).

train.json exists = True bytes = 17277942
val.json exists = True bytes = 2156539
test.json exists = True bytes = 2173858
tools.json exists = True bytes = 4352
taxonomy.json exists = True bytes = 31294
sta

In [ ]:
os.chdir(PROJECT_ROOT)
assert (PROJECT_ROOT / "validate.py").exists(), "validate.py not found in project root."
subprocess.run([sys.executable, "validate.py"], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, "stats.py"], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['/usr/bin/python3', 'stats.py'], returncode=0)

In [ ]:
# Force-reinstall compatible package configurations to avoid backtracking to 5.x versions
%pip install --force-reinstall "transformers==4.47.1" "trl==0.12.2" unsloth unsloth_zoo
%pip install "datasets<5.0.0,>=3.0.0" accelerate bitsandbytes peft

  Using cached transformers-4.47.1-py3-none-any.whl.metadata (44 kB)
  Using cached unsloth-2026.11.10-py3-none-any.whl.metadata (85 kB)
  Using cached unsloth_zoo-2026.11.10-py3-none-any.whl.metadata (33 kB)
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install transformers==4.47.1 and trl==0.12.2 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested transformers==4.47.1
    trl 0.12.2 depends on transformers<4.47.0

Additionally, some packages in these conflicts have no matching distributions available for your environment:
    transformers

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing

In [ ]:
CFG = {
    "run_name": "E2_qwen3_4b_r16_lr2e-4",
    "experiment": "E2",
    "model_name": "unsloth/Qwen3-4B-Instruct-2507",
    "max_seq_length": 2048,
    "load_in_4bit": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    "learning_rate": 2e-4,
    "epochs": 2,
    "per_device_bs": 4,
    "grad_accum": 4,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "scheduler": "cosine",
    "optim": "adamw_8bit",
    "seed": 3407,
    "train_size": 4800,
    "eval_limit": 300,
    "test_limit": 300,
    "max_grad_norm": 1.0,
}
assert CFG["per_device_bs"] * CFG["grad_accum"] == 16, "Baseline effective batch should be 16."
RUNS_DIR = PROJECT_ROOT / "runs"
RUN_DIR = RUNS_DIR / CFG["run_name"]
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "config.json").write_text(json.dumps(CFG, indent=2), encoding="utf-8")
print(json.dumps(CFG, indent=2))

{
  "run_name": "E2_qwen3_4b_r16_lr2e-4",
  "experiment": "E2",
  "model_name": "unsloth/Qwen3-4B-Instruct-2507",
  "max_seq_length": 2048,
  "load_in_4bit": true,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.0,
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "learning_rate": 0.0002,
  "epochs": 2,
  "per_device_bs": 4,
  "grad_accum": 4,
  "warmup_ratio": 0.05,
  "weight_decay": 0.01,
  "scheduler": "cosine",
  "optim": "adamw_8bit",
  "seed": 3407,
  "train_size": 4800,
  "eval_limit": 300,
  "test_limit": 300,
  "max_grad_norm": 1.0
}


In [ ]:
import torch
from datasets import Dataset
from transformers import set_seed

set_seed(CFG["seed"])
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
assert torch.cuda.is_available(), "GPU is not enabled. Select Runtime → Change runtime type → GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

W1124 07:07:50.984000 426 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1124 07:07:51.025000 426 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


GPU: Tesla T4
PyTorch: 2.14.1+cu130
CUDA: 13.0


In [ ]:
import os
import sys
import warnings
import torch

warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Step 1. Force the correct github-main installation of Unsloth first to clear any static limitations
try:
    import unsloth
    # Check if we are running the git main package (which is versioned 2026.11.10 or supports Qwen3-4B-Instruct-2507)
    from unsloth import FastLanguageModel
except (ModuleNotFoundError, ImportError, NotImplementedError):
    print("Installing latest Unsloth development build from GitHub...")
    !pip uninstall unsloth unsloth_zoo -y
    !pip install --no-cache-dir --upgrade "protobuf>=4.21.0" "transformers>=4.47.0,<4.48.0" "trl>=0.13.0,<=0.15.0"
    !pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-cache-dir "git+https://github.com/unslothai/unsloth-zoo.git"
    print("Installation complete. Restarting environment...")
    os.kill(os.getpid(), 9)

# Step 2. If the imports are clean, verify and load the model on GPU
if not torch.cuda.is_available():
    print("CUDA is not visible. Select Runtime -> Change runtime type -> GPU.")
else:
    print("GPU is active:", torch.cuda.get_device_name(0))

    if 'CFG' not in globals() or "lora_r" not in CFG:
        CFG = {
            "model_name": "unsloth/Qwen3-4B-Instruct-2507",
            "max_seq_length": 2048,
            "load_in_4bit": True,
            "lora_r": 16,
            "lora_alpha": 32,
            "lora_dropout": 0.0,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
        }

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=CFG["model_name"],
            max_seq_length=CFG["max_seq_length"],
            dtype=None,
            load_in_4bit=CFG["load_in_4bit"],
        )
    except NotImplementedError:
        print("Re-applying latest github packages and restarting runtime...")
        !pip uninstall unsloth unsloth_zoo -y
        !pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
        !pip install --no-cache-dir "git+https://github.com/unslothai/unsloth-zoo.git"
        os.kill(os.getpid(), 9)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Tokenizer successfully loaded.")
    print("Chat template is active:", tokenizer.chat_template is not None)
    print("EOS Token:", tokenizer.eos_token, "(ID:", tokenizer.eos_token_id, ")")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1124 09:33:33.691000 3877 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1124 09:33:33.731000 3877 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU is active: Tesla T4
==((====))==  Unsloth 2026.11.10: Fast Qwen3 patching. Transformers: 5.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Tokenizer successfully loaded.
Chat template is active: True
EOS Token: <|im_end|> (ID: 151645 )


In [9]:
import sys
import os

# Safely ensure unsloth can be imported, install if missing
try:
    from unsloth import FastLanguageModel
except (ModuleNotFoundError, ImportError):
    import subprocess
    print("Unsloth was not detected in this kernel's path. Reinstalling cleanly...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "unsloth[colab-new]", "unsloth-zoo"], check=True)
    from unsloth import FastLanguageModel

# Safely verify CFG configuration is loaded
if 'CFG' not in globals():
    CFG = {
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.0,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "seed": 3407
    }

# Ensure model is defined. Since the kernel restarted, we may need to reload the model.
if 'model' not in globals() or 'tokenizer' not in globals():
    print("Kernel restarted and 'model' was lost. Reloading Qwen3 model...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CFG.get("model_name", "unsloth/Qwen3-4B-Instruct-2507"),
        max_seq_length=CFG.get("max_seq_length", 2048),
        dtype=None,
        load_in_4bit=CFG.get("load_in_4bit", True),
    )

model = FastLanguageModel.get_peft_model(
    model,
    r=CFG["lora_r"],
    target_modules=CFG["target_modules"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CFG["seed"],
    use_rslora=False,
    loftq_config=None,
)
print("LoRA adapters successfully attached to the Qwen3 model!")

Unsloth: Already have LoRA adapters! We shall skip this step.


LoRA adapters successfully attached to the Qwen3 model!


In [16]:
import json
import random
import zipfile
from pathlib import Path

PROJECT_ROOT = Path("/content/booking_agent")
DATA_ROOT = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# If directory does not exist or files are missing, extract the uploaded zip
if not (DATA_ROOT / "val.json").exists():
    zip_candidates = list(Path("/content").glob("*.zip"))
    if zip_candidates:
        print(f"Extracting {zip_candidates[0].name} to {PROJECT_ROOT}...")
        with zipfile.ZipFile(zip_candidates[0]) as zf:
            zf.extractall(PROJECT_ROOT)
        # Correct the path if there is a nested structure
        if not (DATA_ROOT / "val.json").exists():
            possible = list(PROJECT_ROOT.rglob("val.json"))
            if possible:
                DATA_ROOT = possible[0].parent
                PROJECT_ROOT = DATA_ROOT.parent
    else:
        raise FileNotFoundError("Please make sure 'booking_agent_dataset.zip' exists in /content or upload it again.")

# Ensure CFG contains the required limits
if 'CFG' not in globals():
    CFG = {}
if "eval_limit" not in CFG:
    CFG["eval_limit"] = 300
if "test_limit" not in CFG:
    CFG["test_limit"] = 300

# Reload rows from the extracted folder structure
with open(DATA_ROOT / "val.json", encoding="utf-8") as f:
    val_rows = json.load(f)
with open(DATA_ROOT / "test.json", encoding="utf-8") as f:
    test_rows = json.load(f)

def load_or_create_fixed_ids(path, rows, n, seed):
    path = Path(path)
    if path.exists():
        ids = json.loads(path.read_text(encoding="utf-8"))
        available = {r["id"] for r in rows}
        assert set(ids).issubset(available), f"IDs in {path.name} do not match this dataset."
        return ids
    n = min(n, len(rows))
    ids = [r["id"] for r in random.Random(seed).sample(rows, n)]
    path.write_text(json.dumps(ids, indent=2), encoding="utf-8")
    return ids

VAL_IDS_PATH = RUNS_DIR / "fixed_val_ids.json"
TEST_IDS_PATH = RUNS_DIR / "fixed_test_ids.json"
VAL_IDS = load_or_create_fixed_ids(VAL_IDS_PATH, val_rows, CFG["eval_limit"], 20261124)
TEST_IDS = load_or_create_fixed_ids(TEST_IDS_PATH, test_rows, CFG["test_limit"], 20261124)

val_by_id = {r["id"]: r for r in val_rows}
test_by_id = {r["id"]: r for r in test_rows}
val_subset = [val_by_id[i] for i in VAL_IDS]
test_subset = [test_by_id[i] for i in TEST_IDS]
print("Fixed validation:", len(val_subset), "Fixed test:", len(test_subset))

Fixed validation: 300 Fixed test: 300
